# LO benchmark results

Load multi-model results for the linear-optimization (LO / LP) benchmark into one dataframe, then show a **score table**: rows = questions, columns = LLMs.

Each cell is `1` if the model's JSON `cost` was within 1% of the keyed objective, else `0`.

**Download in this notebook:** the load cell sets `DOWNLOAD_KAGGLE_RUNS = True` and pulls runs via the Kaggle CLI into `data/kaggle_runs/lo-normative-accuracy`. Requires `kaggle auth login` once.

If download is off and files are missing, the cell will still try to download automatically.

Or from a local sandbox / merged CSV: set `LOAD_FROM_KAGGLE = False` and put `*.run.json` or `*merged*.csv` under `data/lp/` or one of the sandbox candidate folders.

In [7]:
# Ensure dependencies for whatever kernel Cursor/VS Code selected.
import importlib.util
import subprocess
import sys

for pkg in ("pandas", "kaggle"):
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg} into: {sys.executable}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
    else:
        print(f"{pkg} available in: {sys.executable}")


pandas available in: c:\src\projects\sceptical_llms\.venv\Scripts\python.exe


In [8]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "lp").is_dir():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import benchmarks.kaggle_runs as kaggle_runs
import benchmarks.lp_rate as lp_rate

importlib.reload(lp_rate)
importlib.reload(kaggle_runs)

from benchmarks.kaggle_runs import (
    DEFAULT_LO_TASK_SLUG,
    download_task_runs,
    load_base_rate_run_rows_from_tree,
    merged_lo_results_from_kaggle_runs,
)
from benchmarks.lp_rate import write_merged_results_csv

# --- knobs ---
LOAD_FROM_KAGGLE = True
DOWNLOAD_KAGGLE_RUNS = True  # download via Kaggle CLI into data/kaggle_runs/...
KAGGLE_TASK_SLUG = DEFAULT_LO_TASK_SLUG  # "lo-normative-accuracy"

# Sandbox / alternate run folders (first existing wins when LOAD_FROM_KAGGLE).
SANDBOX_CANDIDATES = [
    ROOT / "data" / "kaggle_runs" / "lo-normative-accuracy",
    ROOT / "data" / "kaggle_runs" / "lp_benchmark",
    ROOT / "data" / "kaggle_runs" / "lp-benchmark",
    Path("/kaggle/working") / "lp_benchmark",
    Path("/kaggle/working"),
]

BENCHMARK_CSV = ROOT / "data" / "lp" / "benchmark.csv"
MERGED_DIR = ROOT / "data" / "lp"


def _first_existing(paths: list[Path]) -> Path | None:
    for path in paths:
        if path.is_dir() and (
            any(path.rglob("*.run.json")) or any(path.glob("*merged*.csv"))
        ):
            return path
    return None


def _load_merged_from_run_tree(runs_dir: Path) -> tuple[pd.DataFrame, str]:
    try:
        merged_rows = merged_lo_results_from_kaggle_runs(
            runs_dir,
            benchmark_path=BENCHMARK_CSV,
            fill_missing=False,
        )
    except ValueError:
        # Fallback for sandbox trees that do not match the task-slug filter path.
        run_rows = load_base_rate_run_rows_from_tree(runs_dir)
        if not run_rows:
            raise
        out_csv = MERGED_DIR / "lo_merged_results.csv"
        write_merged_results_csv(
            run_rows,
            out_csv,
            benchmark_path=BENCHMARK_CSV,
        )
        merged_rows = pd.read_csv(out_csv).to_dict(orient="records")
    return pd.DataFrame(merged_rows), f"run tree ({runs_dir})"


if LOAD_FROM_KAGGLE:
    default_runs = ROOT / "data" / "kaggle_runs" / KAGGLE_TASK_SLUG
    if DOWNLOAD_KAGGLE_RUNS:
        print(f"Downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)

    runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])
    if runs_dir is None and not DOWNLOAD_KAGGLE_RUNS:
        print(f"No local runs found; downloading {KAGGLE_TASK_SLUG} -> {default_runs}")
        default_runs.mkdir(parents=True, exist_ok=True)
        download_task_runs(KAGGLE_TASK_SLUG, default_runs)
        runs_dir = _first_existing([default_runs, *SANDBOX_CANDIDATES])

    if runs_dir is None:
        raise FileNotFoundError(
            "No LO run results found after download attempt. Looked under:\n  - "
            + "\n  - ".join(str(p) for p in [default_runs, *SANDBOX_CANDIDATES])
            + f"\nManual download:\n  python -m kaggle benchmarks tasks download "
            f"{KAGGLE_TASK_SLUG} -o {default_runs}"
        )

    if any(runs_dir.rglob("*.run.json")):
        df, data_source = _load_merged_from_run_tree(runs_dir)
    else:
        merged_csv = sorted(
            runs_dir.glob("*merged*.csv"),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )[0]
        df = pd.read_csv(merged_csv)
        data_source = str(merged_csv)
else:
    merged_candidates = sorted(
        list(MERGED_DIR.glob("*merged*.csv"))
        + list(MERGED_DIR.glob("lo_merged_results*.csv")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    # De-dupe while preserving mtime order.
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in merged_candidates:
        if path not in seen:
            seen.add(path)
            unique.append(path)
    if not unique:
        raise FileNotFoundError(
            f"Missing merged results under {MERGED_DIR}. "
            "Set LOAD_FROM_KAGGLE=True or run benchmark/lo-benchmark.ipynb."
        )
    merged_csv = unique[0]
    df = pd.read_csv(merged_csv)
    data_source = str(merged_csv)

if "score" not in df.columns:
    raise KeyError("Merged data must include 'score'.")

df["score_value"] = df["score"].astype(str).str.lower().eq("true").astype(int)
if "naive_lp_confusion" in df.columns:
    df["naive_value"] = (
        df["naive_lp_confusion"].astype(str).str.lower().eq("true").astype(int)
    )

print("Source:", data_source)
print("Rows:", len(df))
print("Models:", sorted(df["model"].dropna().unique()))
print("Questions:", sorted(df["example_id"].unique()) if "example_id" in df else "?")
df.head()

RuntimeError: Kaggle CLI not found. Install with: pip install kaggle
Then authenticate: python -m kaggle auth login

## Score table: question × LLM

Rows are LO prompts labeled `vignette_name [condition]`; columns are models. Cell = mean keyed score (`1` = correct within 1%, `0` = incorrect). Implicit and explicit parallels appear as separate rows.

In [ ]:
plot_df = df.copy()
if "condition" in plot_df.columns and "vignette_name" in plot_df.columns:
    plot_df["row_label"] = (
        plot_df["vignette_name"].astype(str)
        + " ["
        + plot_df["condition"].astype(str)
        + "]"
    )
    index_col = "row_label"
elif "vignette_name" in plot_df.columns:
    index_col = "vignette_name"
else:
    index_col = "example_id"

score_table = (
    plot_df.pivot_table(
        index=index_col,
        columns="model",
        values="score_value",
        aggfunc="mean",
    )
    .sort_index()
    .sort_index(axis=1)
)

# Add a mean column / row for a quick overview.
score_table["mean"] = score_table.mean(axis=1)
score_table.loc["mean"] = score_table.mean(axis=0)

display_table = score_table.round(3)
display_table


## Same table keyed by `example_id`

Useful when you want the full prompt id (includes failure mode).

In [ ]:
score_by_id = (
    df.pivot_table(
        index="example_id",
        columns="model",
        values="score_value",
        aggfunc="mean",
    )
    .sort_index()
    .sort_index(axis=1)
)
score_by_id["mean"] = score_by_id.mean(axis=1)
score_by_id.loc["mean"] = score_by_id.mean(axis=0)
score_by_id.round(3)

## Optional: naive-LP confusion (question × LLM)

`1` means the model reported the stated-constraints-only optimum (within 1%).

In [ ]:
if "naive_value" in df.columns:
    naive_df = df.copy()
    if "condition" in naive_df.columns and "vignette_name" in naive_df.columns:
        naive_df["row_label"] = (
            naive_df["vignette_name"].astype(str)
            + " ["
            + naive_df["condition"].astype(str)
            + "]"
        )
        naive_index = "row_label"
    elif "vignette_name" in naive_df.columns:
        naive_index = "vignette_name"
    else:
        naive_index = "example_id"
    naive_table = (
        naive_df.pivot_table(
            index=naive_index,
            columns="model",
            values="naive_value",
            aggfunc="mean",
        )
        .sort_index()
        .sort_index(axis=1)
    )
    naive_table["mean"] = naive_table.mean(axis=1)
    naive_table.loc["mean"] = naive_table.mean(axis=0)
    display(naive_table.round(3))
else:
    print("No naive_lp_confusion column in merged results.")


In [ ]:
# Persist the main score table next to the LP data.
out_path = MERGED_DIR / "lo_score_by_question_model.csv"
score_table.to_csv(out_path)
print("Wrote", out_path)